# Global Temporal Split for NCF

**Input:** `final_baseline_df.pkl`  
**Output:** `App2_train.pkl`, `App2_val.pkl`, `App2_test.pkl`

**Important:** `final_baseline_df.pkl` has already been cleaned upstream:
- Users with exclusively 0-rated interactions are removed
- For users with >= 80% zero-rated interactions, the sequence is clipped to the most recent non-zero rating
- Users still with >= 60% zero-rated interactions after clipping are removed (7,973 users)

**Split logic:**
- All rows in `final_baseline_df.pkl` are used (no further rating filter needed)

**Key Objectives:**

- Maintain chronological integrity to prevent data leakage.
- Evaluate candidate split ratios based on user/item coverage.
- Export full splits and 'warm-start' subsets (seen users and items) for model training.




In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

file_path = '/content/drive/My Drive/BT4222Project/final_updated_baseline_df.pkl'
df = pd.read_pickle(file_path)

Mounted at /content/drive


# Load final_baseline_df

In [ ]:
print(f"Shape:{df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nSample row:")
print(df.head(1).T)

Shape:(8770364, 29)
Columns: ['user_id', 'book_id', 'review_id', 'event_time', 'title', 'description', 'description_truncated', 'has_meaningful_description', 'publication_year', 'num_pages', 'language_code_collapsed', 'format_collapsed', 'is_ebook', 'authors_standardized', 'main_author_id', 'author_interactions_count_before_t', 'top_shelf_tags', 'item_text_for_embedding', 'item_semantic_embedding_static', 'item_num_pages', 'user_page_preference_before_t', 'num_pages_preference_gap', 'user_hist_interaction_count_before_t', 'days_since_user_last_interaction', 'book_interactions_count_before_t', 'days_since_book_last_interaction', 'user_author_interaction_share_before_t', 'user_hist_author_diversity_before_t', 'user_profile_embedding_similarity']

Sample row:
                                                                                        0
user_id                                                  dcdc48c7d14a6467cf277c9c0919eb02
book_id                                              

# Type Coercions and Temporal Sort

In [ ]:
TIME_COL = "event_time"
USER_COL = "user_id"
ITEM_COL = "book_id"

df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce", utc=True)

# Drop rows missing key fields
df = df.dropna(subset=[TIME_COL, USER_COL, ITEM_COL]).copy()

# Optional: standardize id types
df[USER_COL] = df[USER_COL].astype(str)
df[ITEM_COL] = df[ITEM_COL].astype(str)

print("After cleaning:", df.shape)
print("Time range:", df[TIME_COL].min(), "to", df[TIME_COL].max())
print("Unique users:", df[USER_COL].nunique())
print("Unique items:", df[ITEM_COL].nunique())

# ==========================================
# 2) IMPORTANT: sort globally by time
# ==========================================
# Tie-breakers included for deterministic ordering
df = df.sort_values([TIME_COL, USER_COL, ITEM_COL]).reset_index(drop=True)

print("Sorted globally by time.")
print(df[[TIME_COL, USER_COL, ITEM_COL]].head())
print(df[[TIME_COL, USER_COL, ITEM_COL]].tail())

After cleaning: (8770364, 29)
Time range: 1990-01-01 08:00:00+00:00 to 2017-11-04 06:37:07+00:00
Unique users: 454189
Unique items: 123087
Sorted globally by time.
                 event_time                           user_id book_id
0 1990-01-01 08:00:00+00:00  dcdc48c7d14a6467cf277c9c0919eb02   10365
1 1990-01-01 08:00:00+00:00  dcdc48c7d14a6467cf277c9c0919eb02  385247
2 1990-01-01 08:00:00+00:00  dcdc48c7d14a6467cf277c9c0919eb02   39988
3 1990-01-01 08:00:00+00:00  dcdc48c7d14a6467cf277c9c0919eb02    6310
4 1990-01-01 08:00:00+00:00  dcdc48c7d14a6467cf277c9c0919eb02    6319
                       event_time                           user_id   book_id
8770359 2017-11-04 01:45:38+00:00  a12c284783aa16971cee9ccc94517e94     39988
8770360 2017-11-04 02:03:01+00:00  6e4442ce143da1b3d872a7b41f2a7466  28003916
8770361 2017-11-04 02:14:22+00:00  cc2d359563bff087041503c72f3e2f1d      7331
8770362 2017-11-04 03:27:24+00:00  abb217a062dc18e3a02d3df4a755cb78    766955
8770363 2017-11-04 06:37:0

#Global Chronological Split by Row Proportion

*   Opt to split by row proportion instead of time-duration proportion since interactions are heavily concentrated in the later years
*  Identify the split ratio that maintains the best data coverage.



In [ ]:
def evaluate_global_split(
    df: pd.DataFrame,
    train_ratio: float,
    val_ratio: float,
    min_user_prior_list=(1, 3, 5, 10),
):
    test_ratio = 1.0 - train_ratio - val_ratio
    if train_ratio <= 0 or val_ratio <= 0 or test_ratio <= 0:
        raise ValueError("train_ratio, val_ratio, test_ratio must all be positive.")

    n = len(df)
    train_end_idx = int(n * train_ratio)
    val_end_idx = int(n * (train_ratio + val_ratio))

    train_df = df.iloc[:train_end_idx].copy()
    val_df = df.iloc[train_end_idx:val_end_idx].copy()
    test_df = df.iloc[val_end_idx:].copy()

    # Split timestamps
    train_end_time = train_df[TIME_COL].max() if len(train_df) else pd.NaT
    val_end_time = val_df[TIME_COL].max() if len(val_df) else pd.NaT

    # Sets from train only
    train_users = set(train_df[USER_COL].unique())
    train_items = set(train_df[ITEM_COL].unique())

    val_users = set(val_df[USER_COL].unique())
    test_users = set(test_df[USER_COL].unique())
    val_items = set(val_df[ITEM_COL].unique())
    test_items = set(test_df[ITEM_COL].unique())

    # -----------------------------
    # Unique-level seen / unseen
    # -----------------------------
    seen_val_users_unique = len(val_users & train_users)
    unseen_val_users_unique = len(val_users - train_users)

    seen_test_users_unique = len(test_users & train_users)
    unseen_test_users_unique = len(test_users - train_users)

    seen_val_items_unique = len(val_items & train_items)
    unseen_val_items_unique = len(val_items - train_items)

    seen_test_items_unique = len(test_items & train_items)
    unseen_test_items_unique = len(test_items - train_items)

    # Percentages on unique entities
    seen_val_users_unique_pct = seen_val_users_unique / max(len(val_users), 1)
    unseen_val_users_unique_pct = unseen_val_users_unique / max(len(val_users), 1)

    seen_test_users_unique_pct = seen_test_users_unique / max(len(test_users), 1)
    unseen_test_users_unique_pct = unseen_test_users_unique / max(len(test_users), 1)

    seen_val_items_unique_pct = seen_val_items_unique / max(len(val_items), 1)
    unseen_val_items_unique_pct = unseen_val_items_unique / max(len(val_items), 1)

    seen_test_items_unique_pct = seen_test_items_unique / max(len(test_items), 1)
    unseen_test_items_unique_pct = unseen_test_items_unique / max(len(test_items), 1)

    # -----------------------------
    # Row-level seen / unseen
    # -----------------------------
    val_seen_user_rows_pct = val_df[USER_COL].isin(train_users).mean()
    test_seen_user_rows_pct = test_df[USER_COL].isin(train_users).mean()

    val_seen_item_rows_pct = val_df[ITEM_COL].isin(train_items).mean()
    test_seen_item_rows_pct = test_df[ITEM_COL].isin(train_items).mean()

    val_unseen_user_rows_pct = 1.0 - val_seen_user_rows_pct
    test_unseen_user_rows_pct = 1.0 - test_seen_user_rows_pct

    val_unseen_item_rows_pct = 1.0 - val_seen_item_rows_pct
    test_unseen_item_rows_pct = 1.0 - test_seen_item_rows_pct

    # -----------------------------
    # Prior history in train
    # -----------------------------
    train_user_counts = train_df.groupby(USER_COL).size()

    result = {
        "train_ratio": train_ratio,
        "val_ratio": val_ratio,
        "test_ratio": test_ratio,

        "train_rows": len(train_df),
        "val_rows": len(val_df),
        "test_rows": len(test_df),

        "train_users": train_df[USER_COL].nunique(),
        "val_users": val_df[USER_COL].nunique(),
        "test_users": test_df[USER_COL].nunique(),

        "train_items": train_df[ITEM_COL].nunique(),
        "val_items": val_df[ITEM_COL].nunique(),
        "test_items": test_df[ITEM_COL].nunique(),

        "train_end_time": train_end_time,
        "val_end_time": val_end_time,

        # Unique-level user coverage
        "seen_val_users_count": seen_val_users_unique,
        "unseen_val_users_count": unseen_val_users_unique,
        "seen_test_users_count": seen_test_users_unique,
        "unseen_test_users_count": unseen_test_users_unique,

        "seen_val_users_pct": seen_val_users_unique_pct,
        "unseen_val_users_pct": unseen_val_users_unique_pct,
        "seen_test_users_pct": seen_test_users_unique_pct,
        "unseen_test_users_pct": unseen_test_users_unique_pct,

        # Unique-level item coverage
        "seen_val_items_count": seen_val_items_unique,
        "unseen_val_items_count": unseen_val_items_unique,
        "seen_test_items_count": seen_test_items_unique,
        "unseen_test_items_count": unseen_test_items_unique,

        "seen_val_items_pct": seen_val_items_unique_pct,
        "unseen_val_items_pct": unseen_val_items_unique_pct,
        "seen_test_items_pct": seen_test_items_unique_pct,
        "unseen_test_items_pct": unseen_test_items_unique_pct,

        # Row-level coverage
        "val_seen_user_rows_pct": val_seen_user_rows_pct,
        "test_seen_user_rows_pct": test_seen_user_rows_pct,
        "val_unseen_user_rows_pct": val_unseen_user_rows_pct,
        "test_unseen_user_rows_pct": test_unseen_user_rows_pct,

        "val_seen_item_rows_pct": val_seen_item_rows_pct,
        "test_seen_item_rows_pct": test_seen_item_rows_pct,
        "val_unseen_item_rows_pct": val_unseen_item_rows_pct,
        "test_unseen_item_rows_pct": test_unseen_item_rows_pct,
    }

    # Warm-start subset size for different prior thresholds
    for k in min_user_prior_list:
        val_prior = val_df[USER_COL].map(train_user_counts).fillna(0)
        test_prior = test_df[USER_COL].map(train_user_counts).fillna(0)

        val_ws_mask = (
            val_df[USER_COL].isin(train_users)
            & val_df[ITEM_COL].isin(train_items)
            & (val_prior >= k)
        )
        test_ws_mask = (
            test_df[USER_COL].isin(train_users)
            & test_df[ITEM_COL].isin(train_items)
            & (test_prior >= k)
        )

        result[f"val_prior_ge_{k}_rows"] = int(val_ws_mask.sum())
        result[f"test_prior_ge_{k}_rows"] = int(test_ws_mask.sum())
        result[f"val_prior_ge_{k}_pct"] = float(val_ws_mask.mean())
        result[f"test_prior_ge_{k}_pct"] = float(test_ws_mask.mean())

    return result


In [ ]:
results = []

candidate_train_ratios = [0.70, 0.75, 0.80, 0.85, 0.90]
candidate_val_ratios = [0.05, 0.10, 0.15]

for train_ratio in candidate_train_ratios:
    for val_ratio in candidate_val_ratios:
        test_ratio = 1 - train_ratio - val_ratio
        if test_ratio <= 0 or test_ratio < 0.05:
            continue

        res = evaluate_global_split(df, train_ratio, val_ratio)
        results.append(res)

results_df = pd.DataFrame(results)

# -----------------------------
# Rank candidate splits
# Main focus: lower unseen users/items,
# then better warm-start coverage
# -----------------------------
results_df = results_df.sort_values(
    by=[
        "unseen_test_users_pct",     # unique-user cold start
        "unseen_val_users_pct",
        "unseen_test_items_pct",
        "unseen_val_items_pct",
        "test_prior_ge_5_pct",       # larger is better, so descending below
    ],
    ascending=[True, True, True, True, False]
).reset_index(drop=True)

# -----------------------------
# Display summary
# -----------------------------
summary_cols = [
    "train_ratio", "val_ratio", "test_ratio",
    "train_rows", "val_rows", "test_rows",
    "train_end_time", "val_end_time",

    # Unique-level
    "unseen_val_users_count", "unseen_val_users_pct",
    "unseen_test_users_count", "unseen_test_users_pct",
    "unseen_val_items_count", "unseen_val_items_pct",
    "unseen_test_items_count", "unseen_test_items_pct",

    # Row-level
    "val_unseen_user_rows_pct", "test_unseen_user_rows_pct",
    "val_unseen_item_rows_pct", "test_unseen_item_rows_pct",

    # Warm-start
    "val_prior_ge_5_pct", "test_prior_ge_5_pct",
]

print("\nTop candidate splits:")
print(results_df[summary_cols].head(20).to_string(index=False))

# -----------------------------
# Pick best split
# -----------------------------
best = results_df.iloc[0]
print("\nBest split:")
for col in summary_cols:
    print(f"{col:28s}: {best[col]}")



Top candidate splits:
 train_ratio  val_ratio  test_ratio  train_rows  val_rows  test_rows            train_end_time              val_end_time  unseen_val_users_count  unseen_val_users_pct  unseen_test_users_count  unseen_test_users_pct  unseen_val_items_count  unseen_val_items_pct  unseen_test_items_count  unseen_test_items_pct  val_unseen_user_rows_pct  test_unseen_user_rows_pct  val_unseen_item_rows_pct  test_unseen_item_rows_pct  val_prior_ge_5_pct  test_prior_ge_5_pct
        0.85       0.10        0.05     7454809    877036     438519 2016-08-10 22:48:53+00:00 2017-05-16 18:47:10+00:00                   32134              0.269109                    19609               0.265076                    6363              0.086576                     5542               0.098932                  0.253993                   0.275929                  0.051262                   0.147136            0.604886             0.506311
        0.85       0.05        0.10     7454809    438518     877

In [ ]:
# Materialize chosen split

best_train_ratio = float(best["train_ratio"])
best_val_ratio = float(best["val_ratio"])

n = len(df)
train_end_idx = int(n * best_train_ratio)
val_end_idx = int(n * (best_train_ratio + best_val_ratio))

train_df = df.iloc[:train_end_idx].reset_index(drop=True)
val_df = df.iloc[train_end_idx:val_end_idx].reset_index(drop=True)
test_df = df.iloc[val_end_idx:].reset_index(drop=True)

print("\nChosen split sizes:")
print(f"Train: {len(train_df):,}")
print(f"Val  : {len(val_df):,}")
print(f"Test : {len(test_df):,}")


Chosen split sizes:
Train: 7,454,809
Val  : 877,036
Test : 438,519


# Building warm-start subsets for NCF evaluation

In [ ]:
# Warm-start subsets: seen user + seen item
# Then inspect how much extra filtering K adds
train_users = set(train_df[USER_COL].unique())
train_items = set(train_df[ITEM_COL].unique())
train_user_counts = train_df.groupby(USER_COL).size()


# 1) Main warm-start definition
#    only require user/item seen in train
val_seen_mask = (
    val_df[USER_COL].isin(train_users)
    & val_df[ITEM_COL].isin(train_items)
)
test_seen_mask = (
    test_df[USER_COL].isin(train_users)
    & test_df[ITEM_COL].isin(train_items)
)

val_df_ws = val_df.loc[val_seen_mask].reset_index(drop=True)
test_df_ws = test_df.loc[test_seen_mask].reset_index(drop=True)

print("Main warm-start subset (seen user + seen item only):")
print(f"Val WS  : {len(val_df_ws):,} rows ({len(val_df_ws)/len(val_df):.3%} of val)")
print(f"Test WS : {len(test_df_ws):,} rows ({len(test_df_ws)/len(test_df):.3%} of test)")
print()
print("Rows removed by seen-user/item filter:")
print(f"Val removed  : {len(val_df) - len(val_df_ws):,} rows ({1 - len(val_df_ws)/len(val_df):.3%})")
print(f"Test removed : {len(test_df) - len(test_df_ws):,} rows ({1 - len(test_df_ws)/len(test_df):.3%})")

# ------------------------------
# 2) Prior-history diagnostics
#    show how many rows would be
#    additionally removed for each K
# ------------------------------
val_prior = val_df[USER_COL].map(train_user_counts).fillna(0)
test_prior = test_df[USER_COL].map(train_user_counts).fillna(0)

print("\nEffect of adding minimum prior-train-interaction threshold on top of seen-user/item filter:")
for k in [1, 3, 5, 10]:
    val_k_mask = val_seen_mask & (val_prior >= k)
    test_k_mask = test_seen_mask & (test_prior >= k)

    val_k_df = val_df.loc[val_k_mask]
    test_k_df = test_df.loc[test_k_mask]

    val_removed_from_seen = len(val_df_ws) - len(val_k_df)
    test_removed_from_seen = len(test_df_ws) - len(test_k_df)

    print(f"K = {k}")
    print(f"  Val rows kept              : {len(val_k_df):,} ({len(val_k_df)/len(val_df):.3%} of full val)")
    print(f"  Test rows kept             : {len(test_k_df):,} ({len(test_k_df)/len(test_df):.3%} of full test)")
    print(f"  Extra val rows removed     : {val_removed_from_seen:,} ({val_removed_from_seen/max(len(val_df_ws),1):.3%} of seen-user/item subset)")
    print(f"  Extra test rows removed    : {test_removed_from_seen:,} ({test_removed_from_seen/max(len(test_df_ws),1):.3%} of seen-user/item subset)")
    print()

Main warm-start subset (seen user + seen item only):
Val WS  : 612,745 rows (69.865% of val)
Test WS : 259,317 rows (59.135% of test)

Rows removed by seen-user/item filter:
Val removed  : 264,291 rows (30.135%)
Test removed : 179,202 rows (40.865%)

Effect of adding minimum prior-train-interaction threshold on top of seen-user/item filter:
K = 1
  Val rows kept              : 612,745 (69.865% of full val)
  Test rows kept             : 259,317 (59.135% of full test)
  Extra val rows removed     : 0 (0.000% of seen-user/item subset)
  Extra test rows removed    : 0 (0.000% of seen-user/item subset)

K = 3
  Val rows kept              : 564,432 (64.357% of full val)
  Test rows kept             : 236,966 (54.038% of full test)
  Extra val rows removed     : 48,313 (7.885% of seen-user/item subset)
  Extra test rows removed    : 22,351 (8.619% of seen-user/item subset)

K = 5
  Val rows kept              : 530,507 (60.489% of full val)
  Test rows kept             : 222,027 (50.631% of f

# Assert temporal integrity

In [ ]:
assert train_df[TIME_COL].max() <= val_df[TIME_COL].min(), "Train/Val are not globally ordered"
assert val_df[TIME_COL].max() <= test_df[TIME_COL].min(), "Val/Test are not globally ordered"

print("Global temporal ordering check passed.")

Global temporal ordering check passed.


# Save Splits

In [ ]:
save_dir = "/content/drive/My Drive/BT4222Project/"

# Full splits
train_df.to_pickle(os.path.join(save_dir, "App2_train.pkl"))
val_df.to_pickle(os.path.join(save_dir, "App2_val.pkl"))
test_df.to_pickle(os.path.join(save_dir, "App2_test.pkl"))

# Main warm-start subsets: seen user + seen item only
val_df_ws.to_pickle(os.path.join(save_dir, "App2_val_ws.pkl"))
test_df_ws.to_pickle(os.path.join(save_dir, "App2_test_ws.pkl"))

print("Saved successfully:")
print(f"  App2_train.pkl         —  {len(train_df):,} rows  |  {train_df.shape[1]} cols")
print(f"  App2_val.pkl           —  {len(val_df):,} rows  |  {val_df.shape[1]} cols")
print(f"  App2_test.pkl          —  {len(test_df):,} rows  |  {test_df.shape[1]} cols")
print()
print(f"  App2_val_ws.pkl        —  {len(val_df_ws):,} rows  |  {val_df_ws.shape[1]} cols")
print(f"  App2_test_ws.pkl       —  {len(test_df_ws):,} rows  |  {test_df_ws.shape[1]} cols")


Saved successfully:
  App2_train.pkl         —  7,454,809 rows  |  29 cols
  App2_val.pkl           —  877,036 rows  |  29 cols
  App2_test.pkl          —  438,519 rows  |  29 cols

  App2_val_ws.pkl        —  612,745 rows  |  29 cols
  App2_test_ws.pkl       —  259,317 rows  |  29 cols
